In [29]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
import sys; sys.path.append("..")          # agar bisa mengimpor src
import pandas as pd, numpy as np
from src.features import TARGET
from src.cv import cv_oof, report

dev = pd.read_parquet("../data/dev.parquet")

def segment_baseline(tr, va):
    rata2 = tr[TARGET].mean()
    g = tr.groupby(["tenure_bucket", "contract"])[TARGET].mean()
    return np.array([g.get(k, rata2) for k in zip(va["tenure_bucket"], va["contract"])])

oof_seg = cv_oof(dev, segment_baseline)
base_seg = report(dev, oof_seg, "Rata-rata segmen")
print("Acak: PR-AUC ~", round(dev[TARGET].mean(), 3), "| lift@10% = 1.00")

Rata-rata segmen             PR-AUC 0.481 (antar-lipatan 0.456-0.539) | lift@10% 1.95 | brier 0.1580
Acak: PR-AUC ~ 0.265 | lift@10% = 1.00


In [31]:
from src.cv import cv_oof, report, compare_folds
from src.models_lr import make_fit_predict
from src.features import NUM_LR, BIN_LR, CAT_LR

# Versi A: dasar
oofA = cv_oof(dev, make_fit_predict(NUM_LR, BIN_LR, CAT_LR, C=1.0))
assert not np.isnan(oofA).any()
report(dev, oofA, "Logistik A (dasar)")

# Versi B: + fitur gabungan
numB = NUM_LR + ["mtm_x_tenure"]
binB = BIN_LR + ["mtm_x_echeck", "mtm_x_fiber", "mtm_fiber_echeck"]
oofB = cv_oof(dev, make_fit_predict(numB, binB, CAT_LR, C=1.0, interactions=True))
report(dev, oofB, "Logistik B (+gabungan)")

# Perbandingan kelompok demi kelompok
print("\nA vs tebakan segmen"); compare_folds(dev, oofA, oof_seg, "A", "segmen")
print("\nB vs tebakan segmen"); compare_folds(dev, oofB, oof_seg, "B", "segmen")
print("\nB vs A");              compare_folds(dev, oofB, oofA, "B", "A")

Logistik A (dasar)           PR-AUC 0.642 (antar-lipatan 0.623-0.676) | lift@10% 2.76 | brier 0.1372
Logistik B (+gabungan)       PR-AUC 0.645 (antar-lipatan 0.624-0.681) | lift@10% 2.80 | brier 0.1365

A vs tebakan segmen
kelompok 0: A 0.676 vs segmen 0.539  menang
kelompok 1: A 0.645 vs segmen 0.468  menang
kelompok 2: A 0.623 vs segmen 0.480  menang
kelompok 3: A 0.657 vs segmen 0.498  menang
kelompok 4: A 0.628 vs segmen 0.456  menang
=> A menang di 5 dari 5 kelompok

B vs tebakan segmen
kelompok 0: B 0.681 vs segmen 0.539  menang
kelompok 1: B 0.648 vs segmen 0.468  menang
kelompok 2: B 0.624 vs segmen 0.480  menang
kelompok 3: B 0.659 vs segmen 0.498  menang
kelompok 4: B 0.632 vs segmen 0.456  menang
=> B menang di 5 dari 5 kelompok

B vs A
kelompok 0: B 0.681 vs A 0.676  menang
kelompok 1: B 0.648 vs A 0.645  menang
kelompok 2: B 0.624 vs A 0.623  menang
kelompok 3: B 0.659 vs A 0.657  menang
kelompok 4: B 0.632 vs A 0.628  menang
=> B menang di 5 dari 5 kelompok


5

In [32]:
for C in [0.03, 0.1, 0.3, 1, 3]:
    oof = cv_oof(dev, make_fit_predict(numB, binB, CAT_LR, C=C, interactions=True))
    report(dev, oof, f"Logistik B, C={C}")

Logistik B, C=0.03           PR-AUC 0.641 (antar-lipatan 0.621-0.680) | lift@10% 2.77 | brier 0.1372
Logistik B, C=0.1            PR-AUC 0.644 (antar-lipatan 0.623-0.681) | lift@10% 2.78 | brier 0.1366
Logistik B, C=0.3            PR-AUC 0.645 (antar-lipatan 0.624-0.681) | lift@10% 2.80 | brier 0.1365
Logistik B, C=1              PR-AUC 0.645 (antar-lipatan 0.624-0.681) | lift@10% 2.80 | brier 0.1365
Logistik B, C=3              PR-AUC 0.645 (antar-lipatan 0.624-0.680) | lift@10% 2.78 | brier 0.1366


In [33]:
from src.models_lr import build_lr
from src.features import TARGET, add_interactions

C_PILIHAN = 0.3   # ganti sesuai hasil Anda
dev_i = add_interactions(dev)
cols = numB + binB + CAT_LR
m = build_lr(numB, binB, CAT_LR, C=C_PILIHAN).fit(dev_i[cols], dev_i[TARGET])

names = m.named_steps["pre"].get_feature_names_out()
bobot = pd.Series(m.named_steps["lr"].coef_[0], index=names).sort_values()
print(bobot.round(2))

cat__contract_Two year                          -1.12
bin__phone_service                              -0.68
cat__internet_service_No                        -0.60
num__mtm_x_tenure                               -0.41
num__tenure_months                              -0.35
cat__contract_One year                          -0.28
num__n_protection                               -0.28
cat__payment_method_Credit card (automatic)     -0.15
cat__payment_method_Mailed check                -0.15
bin__has_dependents                             -0.12
cat__internet_service_DSL                       -0.11
cat__payment_method_Bank transfer (automatic)   -0.10
bin__mtm_x_echeck                               -0.09
num__charge_vs_hist                             -0.02
bin__has_partner                                -0.01
bin__mtm_fiber_echeck                            0.01
bin__mtm_x_fiber                                 0.02
num__n_streaming                                 0.19
bin__senior_citizen         

In [34]:
from src.models_gbm import (get_categories, apply_categories,
                            make_fit_predict_gbm, gbm_columns)

# Siapkan data: kategori dengan daftar tetap
cats = get_categories(dev)
dev_g = apply_categories(dev, cats)

# Cek: tidak boleh ada nilai kosong di kolom kategori
print(dev_g[list(cats)].isna().sum())
print("Kolom yang dipakai:", len(gbm_columns(dev_g)))
print(gbm_columns(dev_g))

contract            0
payment_method      0
internet_service    0
dtype: int64
Kolom yang dipakai: 23
['tenure_months', 'monthly_charge', 'total_charges', 'avg_monthly_hist', 'charge_vs_hist', 'n_protection', 'n_streaming', 'n_addons', 'contract', 'is_month_to_month', 'payment_method', 'auto_pay', 'is_echeck', 'paperless_billing', 'internet_service', 'has_internet', 'is_fiber', 'phone_service', 'multiple_lines', 'senior_citizen', 'has_partner', 'has_dependents', 'n_family']


In [35]:
from src.models_lr import make_fit_predict
from src.features import NUM_LR, BIN_LR, CAT_LR

numB = NUM_LR + ["mtm_x_tenure"]
binB = BIN_LR + ["mtm_x_echeck", "mtm_x_fiber", "mtm_fiber_echeck"]
oofLR = cv_oof(dev, make_fit_predict(numB, binB, CAT_LR, C=0.3, interactions=True))
report(dev, oofLR, "Logistik B (C=0.3)")

Logistik B (C=0.3)           PR-AUC 0.645 (antar-lipatan 0.624-0.681) | lift@10% 2.80 | brier 0.1365


{'pr_auc': 0.6446529333386719,
 'roc_auc': 0.8436843921228933,
 'lift@10%': np.float64(2.8046668290393675),
 'lift@20%': np.float64(2.543611921324486),
 'brier': 0.136517296812982,
 'logloss': 0.4179808984700238}

In [36]:
oofG = cv_oof(dev_g, make_fit_predict_gbm())
assert not np.isnan(oofG).any()
report(dev_g, oofG, "LightGBM (awal)")
compare_folds(dev, oofG, oofLR, "LightGBM", "Logistik B")

LightGBM (awal)              PR-AUC 0.656 (antar-lipatan 0.637-0.695) | lift@10% 2.82 | brier 0.1348
kelompok 0: LightGBM 0.695 vs Logistik B 0.681  menang
kelompok 1: LightGBM 0.657 vs Logistik B 0.647  menang
kelompok 2: LightGBM 0.637 vs Logistik B 0.624  menang
kelompok 3: LightGBM 0.676 vs Logistik B 0.661  menang
kelompok 4: LightGBM 0.640 vs Logistik B 0.630  menang
=> LightGBM menang di 5 dari 5 kelompok


5

In [37]:
grid = [
    dict(n_estimators=150, num_leaves=4),
    dict(n_estimators=300, num_leaves=4),
    dict(n_estimators=300, num_leaves=8),
    dict(n_estimators=500, num_leaves=8, learning_rate=0.02),
    dict(n_estimators=300, num_leaves=16),
]
hasil = {}
for i, g in enumerate(grid):
    oof = cv_oof(dev_g, make_fit_predict_gbm(**g))
    hasil[i] = oof
    report(dev_g, oof, f"LGBM {g}")

LGBM {'n_estimators': 150, 'num_leaves': 4} PR-AUC 0.651 (antar-lipatan 0.630-0.693) | lift@10% 2.78 | brier 0.1363
LGBM {'n_estimators': 300, 'num_leaves': 4} PR-AUC 0.656 (antar-lipatan 0.637-0.695) | lift@10% 2.82 | brier 0.1348
LGBM {'n_estimators': 300, 'num_leaves': 8} PR-AUC 0.654 (antar-lipatan 0.631-0.688) | lift@10% 2.82 | brier 0.1351
LGBM {'n_estimators': 500, 'num_leaves': 8, 'learning_rate': 0.02} PR-AUC 0.653 (antar-lipatan 0.630-0.687) | lift@10% 2.79 | brier 0.1353
LGBM {'n_estimators': 300, 'num_leaves': 16} PR-AUC 0.647 (antar-lipatan 0.623-0.680) | lift@10% 2.82 | brier 0.1368


In [38]:
oofBest = hasil[1]   # 300 pohon, 4 cabang
report(dev_g, oofBest, "LGBM terpilih (300 pohon, 4 cabang)")
compare_folds(dev, oofBest, oofLR, "LightGBM", "Logistik B")
oofMix = (oofBest + oofLR) / 2
report(dev, oofMix, "Rata-rata LGBM + Logistik")

LGBM terpilih (300 pohon, 4 cabang) PR-AUC 0.656 (antar-lipatan 0.637-0.695) | lift@10% 2.82 | brier 0.1348
kelompok 0: LightGBM 0.695 vs Logistik B 0.681  menang
kelompok 1: LightGBM 0.657 vs Logistik B 0.647  menang
kelompok 2: LightGBM 0.637 vs Logistik B 0.624  menang
kelompok 3: LightGBM 0.676 vs Logistik B 0.661  menang
kelompok 4: LightGBM 0.640 vs Logistik B 0.630  menang
=> LightGBM menang di 5 dari 5 kelompok
Rata-rata LGBM + Logistik    PR-AUC 0.658 (antar-lipatan 0.636-0.695) | lift@10% 2.82 | brier 0.1345


{'pr_auc': 0.6576205943860557,
 'roc_auc': 0.8480266427271059,
 'lift@10%': np.float64(2.824747975786666),
 'lift@20%': np.float64(2.560346210280568),
 'brier': 0.1345028925337107,
 'logloss': 0.4133250920639845}

In [40]:
from src.models_gbm import make_fit_predict_gbm, DEFAULT_PARAMS

oofBest = cv_oof(dev_g, make_fit_predict_gbm(**DEFAULT_PARAMS))

def reliability(y, p, bins=10):
    d = pd.DataFrame({"y": y.values, "p": p})
    d["kelompok"] = pd.qcut(d["p"], bins, duplicates="drop")
    t = d.groupby("kelompok", observed=True).agg(
        n=("y", "size"), rata_prediksi=("p", "mean"), kenyataan=("y", "mean")).round(3)
    t["selisih"] = (t["rata_prediksi"] - t["kenyataan"]).round(3)
    return t

print(reliability(dev["churned"], oofBest))
top = oofBest >= np.quantile(oofBest, 0.9)
print("10% teratas -> prediksi", oofBest[top].mean().round(3),
      "| kenyataan", dev.loc[top, "churned"].mean().round(3))
print("Semua       -> prediksi", oofBest.mean().round(3),
      "| kenyataan", dev["churned"].mean().round(3))

                                  n  rata_prediksi  kenyataan  selisih
kelompok                                                              
(0.008549999999999999, 0.0213]  564          0.017      0.011    0.006
(0.0213, 0.0354]                563          0.027      0.018    0.009
(0.0354, 0.0618]                563          0.047      0.043    0.004
(0.0618, 0.106]                 564          0.082      0.099   -0.017
(0.106, 0.18]                   563          0.143      0.149   -0.006
(0.18, 0.273]                   563          0.227      0.233   -0.006
(0.273, 0.382]                  564          0.323      0.317    0.006
(0.382, 0.525]                  563          0.455      0.442    0.013
(0.525, 0.651]                  563          0.588      0.593   -0.005
(0.651, 0.903]                  564          0.747      0.748   -0.001
10% teratas -> prediksi 0.747 | kenyataan 0.748
Semua       -> prediksi 0.266 | kenyataan 0.265


In [41]:
from src.calibration import cv_oof_calibrated

fp = make_fit_predict_gbm(**BEST_PARAMS)
oof_sig = cv_oof_calibrated(dev_g, fp, "sigmoid")
oof_iso = cv_oof_calibrated(dev_g, fp, "isotonic")

report(dev_g, oofBest, "Tanpa perapian")
report(dev_g, oof_sig, "Sigmoid")
report(dev_g, oof_iso, "Isotonic")

print(reliability(dev["churned"], oof_sig))

Tanpa perapian               PR-AUC 0.656 (antar-lipatan 0.637-0.695) | lift@10% 2.82 | brier 0.1348
Sigmoid                      PR-AUC 0.657 (antar-lipatan 0.637-0.695) | lift@10% 2.82 | brier 0.1348
Isotonic                     PR-AUC 0.652 (antar-lipatan 0.610-0.678) | lift@10% 2.77 | brier 0.1349
                                  n  rata_prediksi  kenyataan  selisih
kelompok                                                              
(0.008990000000000001, 0.0219]  564          0.017      0.011    0.006
(0.0219, 0.036]                 563          0.028      0.018     0.01
(0.036, 0.0627]                 563          0.048      0.041    0.007
(0.0627, 0.107]                 564          0.083      0.101   -0.018
(0.107, 0.181]                  563          0.144      0.151   -0.007
(0.181, 0.274]                  563          0.228      0.234   -0.006
(0.274, 0.382]                  564          0.323      0.314    0.009
(0.382, 0.523]                  563          0.454      0.

In [42]:
from src.cv import fold_ap

kandidat = ["charge_vs_hist", "phone_service", "multiple_lines"]
oof_full = cv_oof(dev_g, make_fit_predict_gbm(**BEST_PARAMS))
ap_full = fold_ap(dev, oof_full)
report(dev_g, oof_full, "LGBM lengkap")

for f in kandidat + [kandidat]:        # satu per satu, lalu ketiganya sekaligus
    drop = f if isinstance(f, list) else [f]
    oof = cv_oof(dev_g, make_fit_predict_gbm(drop=drop, **BEST_PARAMS))
    report(dev_g, oof, "tanpa " + "+".join(drop))
    d = fold_ap(dev, oof) - ap_full
    print("    selisih per kelompok:", d.round(3), f"| rata-rata {d.mean():+.4f}")

LGBM lengkap                 PR-AUC 0.656 (antar-lipatan 0.637-0.695) | lift@10% 2.82 | brier 0.1348
tanpa charge_vs_hist         PR-AUC 0.657 (antar-lipatan 0.640-0.694) | lift@10% 2.83 | brier 0.1348
    selisih per kelompok: [-0.     0.     0.003  0.002 -0.   ] | rata-rata +0.0009
tanpa phone_service          PR-AUC 0.655 (antar-lipatan 0.630-0.697) | lift@10% 2.80 | brier 0.1350
    selisih per kelompok: [ 0.002 -0.003 -0.007  0.     0.001] | rata-rata -0.0014
tanpa multiple_lines         PR-AUC 0.654 (antar-lipatan 0.632-0.695) | lift@10% 2.80 | brier 0.1351
    selisih per kelompok: [ 0.001  0.001 -0.005 -0.002 -0.003] | rata-rata -0.0018
tanpa charge_vs_hist+phone_service+multiple_lines PR-AUC 0.655 (antar-lipatan 0.637-0.698) | lift@10% 2.82 | brier 0.1350
    selisih per kelompok: [ 0.003 -0.001 -0.    -0.002 -0.001] | rata-rata -0.0004


In [43]:
from src.models_lr import make_fit_predict
ap_lr = fold_ap(dev, oofLR)

for f in kandidat:
    nums = [c for c in numB if c != f]
    bins = [c for c in binB if c != f]
    oof = cv_oof(dev, make_fit_predict(nums, bins, CAT_LR, C=0.3, interactions=True))
    report(dev, oof, f"Logistik B tanpa {f}")
    d = fold_ap(dev, oof) - ap_lr
    print("    selisih per kelompok:", d.round(3), f"| rata-rata {d.mean():+.4f}")

Logistik B tanpa charge_vs_hist PR-AUC 0.645 (antar-lipatan 0.626-0.681) | lift@10% 2.78 | brier 0.1365
    selisih per kelompok: [-0.     0.003  0.002 -0.001 -0.001] | rata-rata +0.0007
Logistik B tanpa phone_service PR-AUC 0.643 (antar-lipatan 0.622-0.681) | lift@10% 2.80 | brier 0.1367
    selisih per kelompok: [ 0.     0.    -0.002 -0.003 -0.002] | rata-rata -0.0013
Logistik B tanpa multiple_lines PR-AUC 0.641 (antar-lipatan 0.618-0.682) | lift@10% 2.77 | brier 0.1367
    selisih per kelompok: [ 0.001  0.003 -0.006 -0.001 -0.006] | rata-rata -0.0018


In [44]:
print(pd.crosstab(dev["phone_service"], dev["internet_service"]))
print(dev.groupby(["phone_service", "internet_service"])["churned"]
         .agg(["size", "mean"]).round(3))

internet_service   DSL  Fiber optic    No
phone_service                            
0                  552            0     0
1                 1380         2481  1221
                                size   mean
phone_service internet_service             
0             DSL                552  0.248
1             DSL               1380  0.161
              Fiber optic       2481  0.423
              No                1221  0.071


In [45]:
   from src.models_gbm import gbm_columns
   print("charge_vs_hist" in gbm_columns(dev_g))   # harus False
   print(len(gbm_columns(dev_g)))                  # harus 22

False
22


In [47]:
   oofFinal = cv_oof(dev_g, make_fit_predict_gbm(**DEFAULT_PARAMS))
   report(dev_g, oofFinal, "LGBM final")

   numB = NUM_LR + ["mtm_x_tenure"]
   oofLR = cv_oof(dev, make_fit_predict(numB, binB, CAT_LR, C=0.3, interactions=True))
   report(dev, oofLR, "Logistik B final")

LGBM final                   PR-AUC 0.657 (antar-lipatan 0.640-0.694) | lift@10% 2.83 | brier 0.1348
Logistik B final             PR-AUC 0.645 (antar-lipatan 0.624-0.681) | lift@10% 2.80 | brier 0.1365


{'pr_auc': 0.6446529333386719,
 'roc_auc': 0.8436843921228933,
 'lift@10%': np.float64(2.8046668290393675),
 'lift@20%': np.float64(2.543611921324486),
 'brier': 0.136517296812982,
 'logloss': 0.4179808984700238}

In [48]:
   oofFinal = cv_oof(dev_g, make_fit_predict_gbm(**BEST_PARAMS))
   print(reliability(dev["churned"], oofFinal))
   top = oofFinal >= np.quantile(oofFinal, 0.9)
   print("10% teratas -> prediksi", oofFinal[top].mean().round(3),
         "| kenyataan", dev.loc[top, "churned"].mean().round(3))
   print("Semua       -> prediksi", oofFinal.mean().round(3),
         "| kenyataan", dev["churned"].mean().round(3))

                                  n  rata_prediksi  kenyataan  selisih
kelompok                                                              
(0.007489999999999999, 0.0207]  564          0.016      0.014    0.002
(0.0207, 0.0347]                563          0.027      0.016    0.011
(0.0347, 0.0614]                563          0.047      0.037     0.01
(0.0614, 0.106]                 564          0.082      0.105   -0.023
(0.106, 0.18]                   563          0.142      0.155   -0.013
(0.18, 0.273]                   563          0.226      0.226      0.0
(0.273, 0.384]                  564          0.324       0.31    0.014
(0.384, 0.53]                   563          0.457      0.462   -0.005
(0.53, 0.652]                   563          0.588      0.579    0.009
(0.652, 0.901]                  564          0.747       0.75   -0.003
10% teratas -> prediksi 0.747 | kenyataan 0.75
Semua       -> prediksi 0.265 | kenyataan 0.265
